# Demonstration Calibration Report — HEC-RAS Example 24

This notebook demonstrates the `calibration_report` module using data from the
**HEC-RAS Example Projects** (Example 24: Manning’s n Calibration). All data used here
is distributed with HEC-RAS and available from the
[HEC-RAS Downloads page](https://www.hec.usace.army.mil/software/hec-ras/download.aspx).

**Model:** Mississippi/Ohio River confluence, 1D unsteady flow (Sep 2004–Mar 2005)

The example includes:
- **11 observed gauge stations** from LMRFC in `LMRFC.dss` (8 stage gauges demonstrated here)
- **5 computed plans** in `Manning’snCalibra.dss` (2 plans compared below for brevity)
- Stage time series at 6-hour intervals across Mississippi Upper/Lower and Ohio River reaches

The report generator produces a **self-contained HTML file** with:
- Project introduction and data provenance
- Interactive leaflet map of gauge locations
- Data source and boundary condition tables
- Interactive Bokeh time series plots
- DSS-Commander-style color-coded calibration statistics (RMSE%, Pearson r, PBIAS, NSE, KGE)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# --- User-configured paths (update for your environment) ---
RAS_COMMANDER_PATH = Path(r"G:\GH\ras-commander")
PIPELINE_PATH = Path(r"G:\GH\symphony-workspaces\ras-agent\CLB-423\pipeline")

if str(RAS_COMMANDER_PATH) not in sys.path:
    sys.path.insert(0, str(RAS_COMMANDER_PATH))
if str(PIPELINE_PATH) not in sys.path:
    sys.path.insert(0, str(PIPELINE_PATH))

from ras_commander.dss import RasDss
import calibration_report
from calibration_report import generate_calibration_report, calculate_stats

print(f"RasDss loaded from: {RasDss.__module__}")
print(f"calibration_report loaded from: {calibration_report.__file__}")

## Configuration

Point to the Example 24 project directory and define the simulation window. The gauge-to-station mapping comes from the unsteady flow file (`Manning'snCalibra.u01`).

In [ ]:
# --- User-configured path (update for your environment) ---
EX24_DIR = Path(r"C:\Users\bill\Documents\HEC Data\HEC-RAS\Example Data\Applications Guide\Chapter 24 - Mannings-n-Calibration")

OBSERVED_DSS = str(EX24_DIR / "LMRFC.dss")
MODELED_DSS = str(EX24_DIR / "Manning'snCalibra.dss")

SIM_START = pd.Timestamp("2004-09-25 12:00")
SIM_END = pd.Timestamp("2005-03-05 12:00")

# Subset of plans; add "AUTO CALIBRATE", "SEQUENTIAL", "SQUARED ERROR" to compare all five
PLANS_TO_COMPARE = ["FINAL MODEL", "NOCALIBRATION"]

# Gauge-to-station mapping extracted from Manning'snCalibra.u01
# Each entry: (display_name, gauge_id, observed_dss_pathname, river_reach_part_a, station_part_b)
# Omitted: BRKI2/GCTI2 (pool elevation format), SMLI2 (flow-only gauge with no modeled STAGE match)
GAUGE_MAP = [
    ("Chester IL (CHSI2)",      "CHSI2", "/CHSI2/CHSI2/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI UPPER MISS", "110.4"),
    ("Thebes IL (THBI2)",       "THBI2", "/THBI2/THBI2/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI UPPER MISS", "43.7"),
    ("Price Landing (PCLM7)",   "PCLM7", "/PCLM7/PCLM7/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI UPPER MISS", "28.57"),
    ("Cairo IL (CIRI2)",        "CIRI2", "/CIRI2/CIRI2/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI LOWER MISS.", "953.5"),
    ("Hickman KY (HKMK2)",      "HKMK2", "/HKMK2/HKMK2/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI LOWER MISS.", "922"),
    ("Tiptonville TN (TPTT1)",  "TPTT1", "/TPTT1/TPTT1/STAGE/01SEP2004/6HOUR/OBS/",  "MISSISSIPPI LOWER MISS.", "873"),
    ("Paducah KY (PAHK2)",      "PAHK2", "/PAHK2/PAHK2/STAGE/01SEP2004/6HOUR/OBS/",  "OHIO RIVER OHS",         "-935.66"),
    ("Metropolis IL (MTRI2)",   "MTRI2", "/MTRI2/MTRI2/STAGE/01SEP2004/6HOUR/OBS/",  "OHIO RIVER OHS",         "-944.1"),
]

if not Path(OBSERVED_DSS).exists():
    raise FileNotFoundError(f"Observed DSS not found: {OBSERVED_DSS}\nInstall HEC-RAS Example 24 or update EX24_DIR above.")
if not Path(MODELED_DSS).exists():
    raise FileNotFoundError(f"Modeled DSS not found: {MODELED_DSS}\nInstall HEC-RAS Example 24 or update EX24_DIR above.")

print(f"Project: {EX24_DIR.name}")
print(f"Observed DSS: {Path(OBSERVED_DSS).name} ({Path(OBSERVED_DSS).stat().st_size / 1e6:.1f} MB)")
print(f"Modeled DSS:  {Path(MODELED_DSS).name} ({Path(MODELED_DSS).stat().st_size / 1e6:.1f} MB)")
print(f"Simulation:   {SIM_START} to {SIM_END}")
print(f"Gauges:       {len(GAUGE_MAP)}")
print(f"Plans:        {PLANS_TO_COMPARE}")

## Read Observed Gauge Data

Read each gauge's observed stage time series from `LMRFC.dss` and clip to the simulation window. DSS missing-value sentinels (`-3.4028e+38`) are filtered out.

In [ ]:
def read_dss_series(dss_file, pathname, start, end):
    """Read a DSS time series, filter missing values, and clip to date range."""
    df = RasDss.read_timeseries(dss_file, pathname)
    series = df["value"]
    series = series[series > -1e+37]
    series = series[(series.index >= start) & (series.index <= end)]
    return series

observed_data = {}
for name, gauge_id, obs_path, river_reach, station in GAUGE_MAP:
    try:
        obs = read_dss_series(OBSERVED_DSS, obs_path, SIM_START, SIM_END)
        observed_data[name] = {
            "gauge_id": gauge_id,
            "river_reach": river_reach,
            "station": station,
            "observed": obs,
        }
        print(f"  {name:30s}  {len(obs):5d} pts  range: {obs.min():.1f} - {obs.max():.1f}")
    except Exception as e:
        print(f"  {name:30s}  FAILED: {e}")

print(f"\nLoaded {len(observed_data)} / {len(GAUGE_MAP)} observed gauge series")

## Read Modeled Data

For each gauge location, read the modeled stage from the main DSS for each plan. The DSS pathname pattern is `/{river_reach}/{station}/STAGE/{date}/6HOUR/{plan_name}/`.

In [ ]:
for name, info in observed_data.items():
    river_reach = info["river_reach"]
    station = info["station"]
    info["modeled"] = {}
    
    for plan in PLANS_TO_COMPARE:
        modeled_path = f"/{river_reach}/{station}/STAGE/01SEP2004/6HOUR/{plan}/"
        try:
            mod = read_dss_series(MODELED_DSS, modeled_path, SIM_START, SIM_END)
            info["modeled"][plan] = mod
            print(f"  {name:30s}  {plan:20s}  {len(mod):5d} pts")
        except Exception as e:
            print(f"  {name:30s}  {plan:20s}  FAILED: {e}")

loaded = sum(1 for info in observed_data.values() if info["modeled"])
print(f"\nLoaded modeled data for {loaded} / {len(observed_data)} gauges")

## Preview: Per-Gauge Calibration Statistics

Before generating the full HTML report, compute and display the DSS-Commander-style metrics for each gauge and plan. Thresholds: RMSE% <= 15%, r >= 0.9, PBIAS +/-10%, NSE >= 0.6.

In [ ]:
rows = []
for name, info in observed_data.items():
    obs = info["observed"]
    for plan_label, mod in info.get("modeled", {}).items():
        try:
            stats = calculate_stats(obs, mod, variable="stage")
            rows.append({
                "Gauge": name,
                "Plan": plan_label,
                "N": len(pd.concat([obs.rename("o"), mod.rename("m")], axis=1, join="inner").dropna()),
                "RMSE%": round(stats["rmse_pct"], 1),
                "r": round(stats["pearson_r"], 3),
                "PBIAS%": round(stats["pbias"], 1),
                "NSE": round(stats["nse"], 3),
                "KGE": round(stats["kge"], 3),
            })
        except Exception as e:
            rows.append({"Gauge": name, "Plan": plan_label, "Error": str(e)})

stats_df = pd.DataFrame(rows)
stats_df

## Define Project Context

Build a `ProjectContext` with project description, gauge locations (approximate LMRFC coordinates),
data source metadata, and boundary condition descriptions. This information appears at the top of
the generated HTML report with an interactive leaflet map.

In [ ]:
from calibration_report import ProjectContext

# Approximate WGS84 coordinates for LMRFC gauge stations
GAUGE_COORDS = {
    "Chester IL (CHSI2)":     (37.9017, -89.8220),
    "Thebes IL (THBI2)":      (37.2175, -89.4650),
    "Price Landing (PCLM7)":  (37.0850, -89.3700),
    "Cairo IL (CIRI2)":       (36.9983, -89.1764),
    "Hickman KY (HKMK2)":     (36.5714, -89.1864),
    "Tiptonville TN (TPTT1)": (36.3783, -89.4717),
    "Paducah KY (PAHK2)":     (37.0842, -88.6000),
    "Metropolis IL (MTRI2)":  (37.1511, -88.7320),
}

HEC_RAS_DOWNLOAD_URL = "https://www.hec.usace.army.mil/software/hec-ras/download.aspx"

project_context = ProjectContext(
    title="Demonstration Calibration Report — HEC-RAS Example 24",
    description=(
        "This is a demonstration calibration report using data from the HEC-RAS "
        "Example Projects (Example 24: Manning's n Calibration). The example data "
        "is distributed with HEC-RAS and can be downloaded from the U.S. Army Corps "
        "of Engineers Hydrologic Engineering Center at:\n"
        "https://www.hec.usace.army.mil/software/hec-ras/download.aspx\n\n"
        "Example 24 demonstrates automated Manning's n calibration for a 1D unsteady "
        "flow model of the Mississippi and Ohio River confluence region. The model "
        "covers approximately 130 river miles from Chester, IL (Upper Mississippi) "
        "and Smithland, KY (Ohio River) downstream through Cairo, IL to Tiptonville, "
        "TN (Lower Mississippi).\n\n"
        "The calibration event spans September 2004 through March 2005, capturing a "
        "range of flow conditions including a significant fall/winter flood event. "
        "Five calibration strategies are compared: no calibration (initial estimates), "
        "HEC-RAS automatic calibration, sequential reach-by-reach adjustment, "
        "squared-error minimization, and the final accepted model. This report "
        "compares the Final Model against the uncalibrated baseline."
    ),
    data_sources=[
        {
            "name": "LMRFC Observed Stages",
            "type": "Observed",
            "source": "Lower Mississippi River Forecast Center (NOAA/NWS), distributed with HEC-RAS Example 24",
            "file": "LMRFC.dss",
            "period": "Sep 2004 – Mar 2005",
            "interval": "6-hour",
        },
        {
            "name": "HEC-RAS Computed Stages",
            "type": "Modeled",
            "source": "HEC-RAS 1D Unsteady Flow (5 plans), distributed with HEC-RAS Example 24",
            "file": "Manning'snCalibra.dss",
            "period": "Sep 2004 – Mar 2005",
            "interval": "6-hour",
        },
        {
            "name": "Channel Geometry",
            "type": "Geometry",
            "source": "USACE surveyed cross sections, distributed with HEC-RAS Example 24",
            "file": "Manning'snCalibra.g01",
            "description": "1D cross sections at ~1 mile spacing",
        },
    ],
    boundary_conditions=[
        {
            "name": "Upper Mississippi Inflow",
            "location": "Chester, IL (RM 110.4)",
            "type": "Flow Hydrograph",
            "source": "USGS 07020500 / LMRFC",
        },
        {
            "name": "Ohio River Inflow",
            "location": "Smithland, KY (RM -961)",
            "type": "Flow Hydrograph",
            "source": "USGS 03611500 / LMRFC",
        },
        {
            "name": "Lower Mississippi Downstream",
            "location": "Tiptonville, TN (RM 873)",
            "type": "Stage Hydrograph",
            "source": "LMRFC observed stage",
        },
    ],
    gauge_locations=[
        {"name": name, "lat": lat, "lon": lon}
        for name, (lat, lon) in GAUGE_COORDS.items()
        if name in observed_data
    ],
)

print(f"Project: {project_context.title}")
print(f"Gauge locations: {len(project_context.gauge_locations)}")
print(f"Data sources: {len(project_context.data_sources)}")
print(f"Boundary conditions: {len(project_context.boundary_conditions)}")
print(f"HEC-RAS download: {HEC_RAS_DOWNLOAD_URL}")

## Generate Calibration Report

Build gauge records with coordinates and call `generate_calibration_report()` with the project
context to produce a self-contained HTML file including project introduction, interactive map,
data source and boundary condition tables, Bokeh plots, and color-coded statistics.

In [ ]:
gauge_records = []
for name, info in observed_data.items():
    if not info.get("modeled"):
        continue
    lat, lon = GAUGE_COORDS.get(name, (None, None))
    gauge_records.append({
        "name": name,
        "observed": info["observed"],
        "modeled": info["modeled"],
        "variable": "stage",
        "units": "ft",
        "y": lat,
        "x": lon,
    })

output_path = Path("calibration_report_example24.html")

report_path = generate_calibration_report(
    plan_hdfs=[],
    observed_data=gauge_records,
    output_path=output_path,
    project_context=project_context,
)

file_size_kb = report_path.stat().st_size / 1024
print(f"Report written: {report_path}")
print(f"File size: {file_size_kb:.0f} KB")
print(f"Gauges: {len(gauge_records)}, Plans: {len(PLANS_TO_COMPARE)}")

## Open Report

Open the generated HTML file in the default browser.

In [ ]:
import webbrowser
webbrowser.open(str(report_path.resolve()))
print(f"Opened: {report_path.resolve()}")